# My Finding Z analysis
**Question:** *What are you trying to learn? Write one sentence here.*

Run the cells from the top. Edit the short choices cells, then rerun the cells below them. Save this notebook with your plots and written conclusions; follow your assignment's submission instructions.

Select **Python (hep)**. Run **Run → Run All Cells** for an initial pass, then edit the choices below. Run the setup cell as provided; edit the choices below it.

In [ ]:
from findingz.notebook_analysis import (
    available_samples, sample_table, select_samples,
    plot_samples, compare_samples, comparison_table,
)

## 1. Choose samples
The table lists the samples you can use. Copy their IDs into `sample_ids` below. A notebook exported from FindingZ starts with your existing choices.

In [ ]:
analysis = {"plot": None, "count": None}

In [ ]:
library = available_samples()
sample_table(library)

In [ ]:
saved_plot = analysis.get("plot") or {}
sample_ids = saved_plot.get("samples", [])  # e.g. ["run:your-run-id"]
observable = saved_plot.get("observable", "mll")
shape_only = not saved_plot.get("expected_yields", False)
luminosity_fb = saved_plot.get("luminosity_fb", 1.0)

## 2. Choose cuts and plot
`cuts` maps a variable to its allowed range, for example `{"mll": (80, 100)}` in GeV. An empty dictionary means no additional range cuts. `channels = None` keeps all channels; an empty list selects none.

Shape-only plots compare distributions, not event rates. Turn `shape_only` off to use cross sections and luminosity.

In [ ]:
cuts = saved_plot.get("windows", {})  # try {"mll": (80, 100)}
channels = saved_plot.get("channels", None)
cuts  # inspect the cuts before applying them

In [ ]:
frames = select_samples(
    library, sample_ids, cuts=cuts, channels=channels,
    luminosity_fb=None if shape_only else luminosity_fb,
    variables=saved_plot.get("variables"),
)
figure, yields = plot_samples(
    frames, library, observable, shape_only=shape_only,
    variables=saved_plot.get("variables"),
)
if figure is not None:
    display(figure)
else:
    print("Choose sample IDs above to make a plot.")
display(yields)

**What do you notice?**

*Describe the distribution. Change one cut and explain what changed—not just whether the plot looks different.*

## 3. Compare two complete predictions
Choose a **null prediction** and a **complete alternative prediction** for the same final states and collider configuration. The alternative already includes backgrounds; the two samples are never added.

These cuts are independent of the plot cuts. Set `count_cuts = dict(cuts)` to copy them.

Compare the alternative minus the null. A positive difference is an excess, a negative difference a deficit. Expected sensitivity uses the signed square root of the Asimov Poisson likelihood-ratio statistic. With a known null, its magnitude is sqrt(2 × [N₁ ln(N₁/N₀) − N₁ + N₀]). A nonzero null uncertainty profiles the null rate with a Gaussian constraint of width δ₀N₀. This is not an observed-data significance or an exact Poisson p-value; small counts need care. Finite-simulation uncertainty is not included.

In [ ]:
saved_count = analysis.get("count") or {}
null_id = saved_count.get("null")
alternative_id = saved_count.get("alternative")
count_cuts = saved_count.get("windows", {})
count_channels = saved_count.get("channels")
count_luminosity_fb = saved_count.get("luminosity_fb", 1.0)
null_uncertainty = saved_count.get("null_uncertainty_fraction", 0.0)

In [ ]:
result = compare_samples(
    library, null_id, alternative_id,
    cuts=count_cuts, channels=count_channels,
    luminosity_fb=count_luminosity_fb,
    null_uncertainty=null_uncertainty,
    variables=saved_count.get("variables"),
)
comparison_table(result)

## 4. Your conclusion
*What does this analysis establish? Which assumption or uncertainty matters most?*

**Before submitting:** save the notebook with its outputs. If FindingZ created a matching `.settings.json` file, download and submit it alongside the notebook; it preserves the original settings and variable definitions. Event files are not embedded and must remain available.

### Optional extension: redefine a lepton
The plots above use the saved event table. Object-level cuts are a separate exercise: changing a reconstructed lepton definition does not automatically rebuild that table. For a full-pipeline run with retained ROOT, start below or use the lepton-definition exercise.

In [ ]:
# from findingz.delphes import open_run
# events = open_run("YOUR_RUN_ID")  # raw run ID, without "run:"
# muons = events.muons
# selected_muons = muons[(muons.pt > 20) & (abs(muons.eta) < 2.4)]
# Next: construct an opposite-sign pair and its invariant mass.